In [ ]:
import pandas as pd
import requests
import time
import os

# Create a directory to save the CSV files
output_dir = "Billboard_Top_100_CSVs"
os.makedirs(output_dir, exist_ok=True)

# Loop through the requested years
for year in range(1990, 2018):
    print(f"Fetching data for {year}...")
    url = f"https://en.wikipedia.org/wiki/Billboard_Year-End_Hot_100_singles_of_{year}"
    
    try:
        # Use pandas to read all tables on the Wikipedia page
        tables = pd.read_html(url)
        
        # Find the table that contains the Billboard Hot 100 list
        # We look for tables that have a "Title" or "Song" column
        df = None
        for table in tables:
            # Normalize column names to title case for easier matching
            table.columns = [str(col).title() for col in table.columns]
            
            # The column is usually named 'Title' but sometimes 'Song'
            if 'Title' in table.columns and 'Artist(S)' in table.columns:
                df = table[['Title', 'Artist(S)']]
                break
            elif 'Title' in table.columns and 'Artist' in table.columns:
                df = table[['Title', 'Artist']]
                break
            
        if df is not None:
            # Rename columns to match your request
            df.columns = ['Song Name', 'Artist Name']
            
            # Clean up the text (removes Wikipedia citation brackets like [1])
            df['Song Name'] = df['Song Name'].str.replace(r'\[.*?\]', '', regex=True).str.strip()
            df['Song Name'] = df['Song Name'].str.replace('"', '') # Remove quotation marks
            df['Artist Name'] = df['Artist Name'].str.replace(r'\[.*?\]', '', regex=True).str.strip()
            
            # Save to CSV
            filename = os.path.join(output_dir, f"Billboard_Top_100_{year}.csv")
            df.to_csv(filename, index=False, encoding='utf-8')
            print(f"Successfully saved {filename}")
        else:
            print(f"Could not find the correct table for {year}.")
            
    except Exception as e:
        print(f"Error fetching data for {year}: {e}")
        
    # Be polite to Wikipedia's servers
    time.sleep(1)

print("\nAll done! Check the folder:", output_dir)